# SAA Challenger Optimiser (simple version)

**Purpose:** sense-check the developer's SAA optimisation by rebuilding it in a simple, transparent way.

**What the developer's model does (RISKOptimizer, per the SAA Instructions):**
- Chooses 7 composite weights (integers 0–1000, rescaled to sum to 100%).
- For each candidate portfolio, simulates 20 years of correlated returns and computes the **mean portfolio value in year 20** (cell W97).
- Picks the portfolio with the highest mean value, after subtracting **penalties** for breaking constraints:
  - (1000 × breach)² for volatility and minimum yield
  - (100 × breach)² for the asset-class min/max limits

**What this notebook does — two challengers for each strategy:**
1. **Replica** – the same recipe as above (simulation + penalties + genetic-style search).
2. **Analytical** – a direct calculation with no simulation: maximise expected return subject to the constraints.

**Why the analytical result is a fair benchmark:** with annual rebalancing and independent years, the mean value in year 20 equals
Initial value × (1 + expected portfolio return)²⁰. So *maximising mean year-20 value is the same as maximising expected return*. The replica should land close to the analytical answer; any gap comes from simulation noise, the penalty design, or an incomplete search.

Sections: **1. Setup · 2. Input Data Reading · 3. Optimisation · 4. Output Creation**

## 1. Setup
Edit the cells in this section only.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import saa_helpers as h          # the helper functions (saa_helpers.py, same folder)

pd.set_option("display.float_format", "{:.4f}".format)

In [ ]:
# --- Files ---
INPUT_FILE  = "SAA_Challenger_Inputs.xlsx"     # the template
OUTPUT_FILE = "SAA_Challenger_Results_simple.xlsx"

# --- Which strategies to run (column names from the Setup tab); None = all ---
STRATEGIES = None                  # e.g. ["Strategy_3", "Strategy_5"]

# --- Search effort ---
# Each generation tests (PopulationPerAsset x 7) portfolios. 300 generations x 35 = ~10,500 trials.
GENERATIONS = 300
FINAL_ITERATIONS = 10000           # iterations for the final simulation of each portfolio

# --- Developer's results, for comparison (optional) ---
# Enter the weights from the developer's SAA Output, in the order:
# Equity, DMGovt, BondOther, Property, Gold, HFs, Cash   (as decimals, summing to 1)
DEVELOPER_WEIGHTS = {
    # "Strategy_3": [0.45, 0.00, 0.35, 0.10, 0.10, 0.00, 0.00],
}

## 2. Input Data Reading

In [ ]:
inputs = h.read_inputs(INPUT_FILE)

summary_inputs = pd.DataFrame({
    "Expected return": inputs["returns"],
    "Volatility": inputs["vols"],
    "Yield": inputs["yields"],
}).loc[h.ALL_VARS]                 # keep the template order
summary_inputs

In [ ]:
print("Correlation matrix (rank / Spearman):")
display(inputs["corr"].round(2))
h.check_correlation(inputs["corr"])

In [ ]:
print("Constraints per strategy:")
display(inputs["setup"])

print("Settings read from the template:")
for k, v in inputs["settings"].items():
    print(f"  {k:24s} {v}")

In [ ]:
print("Composite weights (index level) - totals per composite should be 100%:")
display(inputs["asset_map"])
inputs["asset_map"].groupby("Composite")["Weight"].sum()

## 3. Optimisation

For each strategy:
1. **Replica, pass 1** – simulate 500 iterations, search for the best penalised mean year-20 value.
2. **Replica, pass 2** – simulate 1,000 new iterations and search again, starting from the pass-1 answer (as in the Instructions).
3. **Analytical** – maximise expected return directly, with hard constraints.

Expect roughly 5–10 seconds per strategy.

In [ ]:
st = inputs["settings"]
strategies = STRATEGIES or list(inputs["setup"].columns)

replica_weights = {}
analytical_weights = {}

for s in strategies:
    constraints = inputs["setup"][s]
    print(f"{s}: optimising ...")

    raw1, w1 = h.optimise_replica(inputs, constraints, int(st["Pass1Iterations"]), seed=1,
                                  generations=GENERATIONS)
    raw2, w2 = h.optimise_replica(inputs, constraints, int(st["Pass2Iterations"]), seed=2,
                                  start=raw1, generations=GENERATIONS)
    replica_weights[s] = w2
    analytical_weights[s] = h.optimise_analytical(inputs, constraints)

    print(f"   pass 1 raw integers: {raw1.astype(int)}")
    print(f"   pass 2 raw integers: {raw2.astype(int)}")
print("Done.")

## 4. Output Creation

### 4.1 Optimal weights – Replica vs Analytical (vs Developer, if entered)

In [ ]:
rows = {}
for s in strategies:
    rows[(s, "Replica")] = replica_weights[s]
    if analytical_weights[s] is not None:
        rows[(s, "Analytical")] = analytical_weights[s]
    if s in DEVELOPER_WEIGHTS:
        rows[(s, "Developer")] = np.array(DEVELOPER_WEIGHTS[s])

weights_table = pd.DataFrame(rows, index=h.ASSETS)     # assets down, (strategy, method) across
weights_table.T.style.format("{:.1%}")

### 4.2 Portfolio statistics (final simulation, 10,000 iterations)
All portfolios are tested on the **same** simulated scenarios, so differences are due to the weights only.

In [ ]:
final_sims = h.simulate_returns(inputs, FINAL_ITERATIONS, int(st["HorizonYears"]),
                                seed=99, distribution=st["Distribution"])

stats_rows = {}
for (s, method) in weights_table.columns:
    stats_rows[(s, method)] = h.portfolio_stats(weights_table[(s, method)].values,
                                                inputs, inputs["setup"][s], final_sims)
stats_table = pd.DataFrame(stats_rows).T
stats_table

**How to read this:**
- *Replica vs Analytical:* if the replica's expected return is slightly **higher** and volatility slightly **above** target, the optimiser is "paying" a small penalty to buy extra return – a direct effect of using soft constraints.
- *Developer vs Analytical:* a developer portfolio with materially lower expected return at the same volatility suggests the RISKOptimizer search may have stopped early, or that its inputs/constraints differ from those in the template.

### 4.3 Risk–return picture

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# individual composites
for a in h.ASSETS:
    ax.scatter(inputs["vols"][a], inputs["returns"][a], color="grey", marker="s")
    ax.annotate(a, (inputs["vols"][a], inputs["returns"][a]), fontsize=8, xytext=(4, 2),
                textcoords="offset points")

# optimised portfolios
markers = {"Replica": "o", "Analytical": "x", "Developer": "*"}
for (s, method), row in stats_table.iterrows():
    ax.scatter(row["Volatility (analytic)"], row["Expected return"], marker=markers[method], s=70,
               label=method)
    ax.annotate(s, (row["Volatility (analytic)"], row["Expected return"]), fontsize=7,
                xytext=(4, -8), textcoords="offset points")

handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys())
ax.set_xlabel("Volatility")
ax.set_ylabel("Expected return")
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.1%}")
ax.set_title("Composites and optimised portfolios")
plt.show()

### 4.4 Weights by strategy (bar chart)

In [ ]:
weights_table.T.plot(kind="bar", stacked=True, figsize=(11, 5), colormap="tab10")
plt.ylabel("Weight")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.title("Composite weights by strategy and method")
plt.tight_layout()
plt.show()

### 4.5 Look-through to the 19 indices

In [ ]:
flat = weights_table.copy()
flat.columns = [f"{s} | {m}" for s, m in flat.columns]
lookthrough = h.look_through(flat, inputs["asset_map"])
lookthrough

### 4.6 Save to Excel

In [ ]:
h.save_results(OUTPUT_FILE, {
    "Weights": flat.T,
    "Statistics": stats_table,
    "LookThrough": lookthrough.set_index(["Composite", "Index"]),
    "Inputs": summary_inputs,
})